In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("inspection_manual.pdf")

docs = loader.load()

print(len(docs))

C:\Users\DELL\AppData\Local\Temp\ipykernel_25760\843939694.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


25


In [5]:
import langchain
print(langchain.__version__)

1.3.13


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

documents = text_splitter.split_documents(docs)

print("Total Chunks:", len(documents))

Total Chunks: 214


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1090.57it/s]


Embeddings model loaded successfully!


In [7]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory="vector_db"
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [8]:
query = input("Enter your query regarding PCB Inspection: ")

docs = db.similarity_search(query)

for i in range(len(docs)):
    print(f"\nResult {i+1}:")
    print(docs[i].page_content)
    print("-" * 80)


Result 1:
is placed, and is sometimes held in place by an adhesive. The entire board is then loaded into an infrared or 
nitrogen oven and “baked”. The solder paste melts (reflows) on the pads and component leads to make the joint. 
A newer reflow method called pin-in-paste or intrusive reflow is available for through hole devices. 
Combinations of wave and reflow soldering can be used for mixed through hole and surface mount boards.
--------------------------------------------------------------------------------

Result 2:
Soldering................................ ................................ ................................ ................................ ............. 22 
Basic PCB Manufacture................................ ................................ ................................ ........................ 23 
Surface Finishies ................................ ................................ ................................ ................................ . 24
------

In [9]:
context = ""

for doc in docs:
    context += doc.page_content + "\n\n"

In [10]:
prompt = f"""
You are an Electronics Engineering Assistant.

Answer ONLY from the given context.

If the answer is not found in the context, reply:

"I couldn't find this information in the uploaded document."

Context:
{context}

Question:
{query}
"""
import ollama

response = ollama.chat(
    model="gemma2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

Soldering considerations need to taken into account when laying out your board.  There are three basic soldering techniques - hand, wave, and reflow. 
Hand soldering is the traditional method typically used f or prototypes and small production runs. Major impacts  when laying out your board include suitable access for the iron, and thermal reflief for pads. Non-plated through




